In [5]:
import pandas as pd
from config import *
import glob
import os

In [127]:
files = glob.glob(f"{ROOT}/data/raw/sectors/*.csv")

In [128]:
sector_mapping = {
    'Utilities_daily.csv': 'utilities',
    'Consumer Staples_daily.csv': 'consumer_staples',
    'Financials_daily.csv': 'financials',
    'Communication Services_daily.csv': 'communication_services',
    'Technology_daily.csv': 'technology',
    'Materials_daily.csv': 'materials',
    'Energy_daily.csv': 'energy',
    'Real Estate_daily.csv': 'real_estate',
    'Consumer Discretionary_daily.csv': 'consumer_discretionary',
    'Industrials_daily.csv': 'industrials',
    'Healthcare_daily.csv': 'healthcare',
}

In [135]:
def sector_cleaning(file, col=None, start_year= None):
    
    df = pd.read_csv(file).drop(index=[0,1]).reset_index(drop=True)
    df.columns = df.columns.str.lower()
    df.rename(columns={
        df.columns[0]: 'date',
    }, inplace=True)
    df['date'] = pd.to_datetime(df['date'])
    cols = df.columns[1:]
    df[cols] = df[cols].astype('float')
    df[col] = round((df['close'].pct_change() * 100), 3)

    if start_year is not None:
        df = df[df['date'].dt.year >= start_year]
    
    if col is not None:    
        df_clean = df[['date', col]]
        return df_clean

    return df


def sector_combining(files):
    df_clean = None
    
    for file in files:
        filename = os.path.basename(file)
        df = sector_cleaning(file, col=sector_mapping[filename])
        if df_clean is None:
            df_clean = df
        else:
            df_clean = pd.merge(df_clean, df, on='date', how='outer')
                
    return df_clean 

        

In [160]:
df_daily = sector_combining(files)
df_monthly = df_daily.set_index('date').resample('ME').first().reset_index()
df_quarter = df_daily.set_index('date').resample('QE').first().reset_index()